# 1. Import Library & Configuration

In [2]:
import pandas as pd
import torch
import chromadb
import numpy as np
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

BASE_DIR = Path.cwd().parents[1]
DATA_DIR = BASE_DIR / 'data'
MODEL_DIR = BASE_DIR / 'models' / 'distilbert_prod'

DB_DIR = BASE_DIR / 'chroma_db'
DB_DIR.mkdir(exist_ok=True)

df_movies = pd.read_csv(DATA_DIR / 'tmdb_movies_with_emotions.csv')

c:\Users\VICTUS\OneDrive\Documents\Edwin's_Project\Moofy\emovie\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Vector Database (ChromaDB)

In [3]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))
collection = chroma_client.get_or_create_collection(name="movie_synopses")

c:\Users\VICTUS\OneDrive\Documents\Edwin's_Project\Moofy\emovie\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VICTUS\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2499.47it

In [4]:
if collection.count() == 0:    
    docs = df_movies['overview'].astype(str).tolist()
    ids = df_movies['movie_id'].astype(str).tolist()

    metadatas = df_movies.apply(
        lambda row: {'title': row['title'], 'emotion': row['emotion_label']}, 
        axis=1
    ).tolist()

    embeddings = sbert_model.encode(docs, show_progress_bar=True).tolist()

    collection.add(documents=docs, embeddings=embeddings, metadatas=metadatas, ids=ids)
    print("Successfully saved permanently!")
else:
    print(f"VectorDB alrdy contains {collection.count()} film")


Batches: 100%|██████████| 31/31 [00:10<00:00,  2.95it/s]


Successfully saved permanently!


# 3. Load DistilBERT

In [5]:
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_DIR)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)

with open(MODEL_DIR / 'label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2425.49it/s]


# 4. Recommendation Engine

In [8]:
def get_movie_recommendations(user_text, n_results=5):
    inputs = tokenizer(
        user_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    pred_id = np.argmax(outputs.logits.numpy(), axis=1)[0]
    detected_emotion = label_encoder.inverse_transform([pred_id])[0]

    user_embedding = sbert_model.encode(user_text).tolist()

    results = collection.query(
        query_embeddings=[user_embedding],
        n_results=n_results,
        where={"emotion": detected_emotion}
    )

    recommendations = []

    for i in range(len(results['ids'][0])):
        title = results['metadatas'][0][i]['title']
        overview = results['documents'][0][i]
        distance = results['distances'][0][i] 
        
        print(f"{i+1}. {title} (Similarity Score: {distance:.4f})")
        print(f"   Sinopsis: {overview}\n")

In [9]:
# Tes dengan berbagai macam kalimat!
curhatan_1 = "I am so overwhelmed with my college assignments, I feel like I'm drowning and can't breathe."
get_movie_recommendations(curhatan_1)

print("-" * 50)

curhatan_2 = "Today is the best day of my life, I just won a competition and I want to celebrate!"
get_movie_recommendations(curhatan_2)

1. Inside Out 2 (Similarity Score: 1.3566)
   Sinopsis: Teenager Riley's mind headquarters is undergoing a sudden demolition to make room for something entirely unexpected: new Emotions! Joy, Sadness, Anger, Fear and Disgust, who’ve long been running a successful operation by all accounts, aren’t sure how to feel when Anxiety shows up. And it looks like she’s not alone.

2. Superbad (Similarity Score: 1.4554)
   Sinopsis: Two co-dependent high school seniors are forced to deal with separation anxiety after their plan to stage a booze-soaked party goes awry.

3. The Whale (Similarity Score: 1.5914)
   Sinopsis: A reclusive English teacher suffering from severe obesity attempts to reconnect with his estranged teenage daughter for one last chance at redemption.

4. Dawn of the Planet of the Apes (Similarity Score: 1.6163)
   Sinopsis: A group of scientists in San Francisco struggle to stay alive in the aftermath of a plague that is wiping out humanity, while Caesar tries to maintain domin